# ComfyUI PromptDB — Kaggle + Google Drive + Cloudflare

Run all cells in order. The defaults enable Google Drive synchronization, Krea2 model downloads, Cloudflare Quick Tunnel, and automatic Output backup.

Before running, attach the existing Kaggle Secret `RCLONE_CONFIG_TEXT` from **Add-ons → Secrets**. Its value must be an rclone config containing Google Drive OAuth credentials. A single Google Drive remote is normalized to `gdrive:`; the OAuth scope must allow read and write access (`drive`). The notebook never prints credential values.

Drive data lives under `gdrive:ComfyUI-PromptDB`. Output and runtime data are backed up every two minutes. Cloudflare Quick Tunnel creates a temporary public ComfyUI URL; keep the notebook running while using it. Kaggle can recycle or time out runtimes, so periodic backups are used.

The notebook checks the local ComfyUI API and public Tunnel route. It does not run an image-generation test.

In [ ]:
# 1. Options, repository checkout, rclone/Google Drive setup, and ComfyUI install
from pathlib import Path
import importlib.util
import subprocess

STORE_MODELS_ON_DRIVE = True
DOWNLOAD_KREA2_MODELS = True
FORCE_VERIFY_MODELS = False
ENABLE_CLOUDFLARE_TUNNEL = True
BACKUP_OUTPUT = True
DRIVE_FOLDER = "ComfyUI-PromptDB"
RCLONE_SECRET_NAME = "RCLONE_CONFIG_TEXT"
BACKUP_INTERVAL_SECONDS = 120
REPO_BRANCH = "codex/kaggle-promptdb"
REPO_URL = "https://github.com/JackChen6203/Google-Colab_Notebooks.git"
REPO = Path("/kaggle/temp/Google-Colab_Notebooks")

if REPO.exists() and (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)

module_path = REPO / "promptdb_colab" / "kaggle_runtime.py"
spec = importlib.util.spec_from_file_location("promptdb_kaggle_runtime", module_path)
if spec is None or spec.loader is None:
    raise RuntimeError("Could not load PromptDB Kaggle runtime from the selected GitHub branch.")
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)

context = runtime.bootstrap(
    REPO,
    drive_folder=DRIVE_FOLDER,
    rclone_secret_name=RCLONE_SECRET_NAME,
    store_models_on_drive=STORE_MODELS_ON_DRIVE,
    download_krea2_models=DOWNLOAD_KREA2_MODELS,
    force_verify_models=FORCE_VERIFY_MODELS,
)

## 2. Start ComfyUI, Cloudflare Tunnel, and Drive backups

This cell waits for the local ComfyUI API, starts a Cloudflare Quick Tunnel, and checks the public UI and PromptDB API. The generated URL is temporary and public.

In [ ]:
# 2. Start ComfyUI, Cloudflare Tunnel, and automatic Drive backups
COMFYUI_URL = runtime.start_services(
    context,
    enable_cloudflare=ENABLE_CLOUDFLARE_TUNNEL,
    backup_output=BACKUP_OUTPUT,
    backup_interval=BACKUP_INTERVAL_SECONDS,
)

## 3. Monitor the ComfyUI queue

Keep this cell active while using ComfyUI. Queue changes and service diagnostics are saved under `gdrive:ComfyUI-PromptDB/logs`. Stop the cell normally to request a final Drive backup; Kaggle can still terminate a runtime without a final callback.

In [ ]:
# 3. Monitor the ComfyUI queue and keep the services active
runtime.monitor(context)